<a href="https://colab.research.google.com/github/SheffieldML/notebook/blob/main/GPy/inla_vs_hmc_poisson.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# INLA versus HMC on a Rue-style latent Gaussian model

Håvard Rue and colleagues introduced INLA for *latent Gaussian models*: a Gaussian process (or Markov random field) $f$ enters a non-Gaussian likelihood, and a small number of hyperparameters $\theta$ control the process. Their canonical examples — Tokyo rainfall, Scottish lip cancer, and the coal mining disasters series — all share this structure. INLA treats $f$ with a Laplace approximation and integrates over $\theta$ numerically on a small design. We follow the same recipe here with `INLAHyperparameters` from CIP-0015.

The question is whether the design-based integration over $\theta$ agrees with a sampler that targets the *same* hyperparameter posterior. We use a series in the style of the coal mining disasters data: yearly counts $y_t \sim \text{Poisson}(\exp f(t))$ over 112 years, with a rate that drops abruptly. We simulate it rather than load the historical series, so that we know the truth. The latent function has a Matérn-3/2 prior, a rough process that suits a sharp change, with log-Gaussian priors on its variance and lengthscale.

Two things differ from the earlier `inla_vs_hmc.ipynb` (in `cip/cip0015` of the GPy repository), which used Bernoulli data with an RBF kernel. Here the likelihood is Poisson and the truth is known, so we can also ask how the latent-rate posterior compares with the generating rate. We also report cost.

As before, both methods share the Laplace marginal likelihood. HMC samples $\theta$ under it, so any disagreement comes from the integration over $\theta$, not from the Laplace approximation of $f$.

In [ ]:
try:
    from GPy.inference.hyperparameters import INLAHyperparameters
except ImportError:
    # INLAHyperparameters is on GPy main (CIP-0015) and may not be in the latest PyPI release.
    import subprocess
    import sys
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "GPy[plotting] @ git+https://github.com/SheffieldML/GPy.git@main"]
    )
    from GPy.inference.hyperparameters import INLAHyperparameters

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
import GPy
from GPy.inference.hyperparameters import INLAHyperparameters

rng = np.random.default_rng(3)
years = np.arange(1851, 1963)
true_rate = 0.9 + 2.2 / (1 + np.exp((years - 1890) / 4.0))   # abrupt drop around 1890
Y = rng.poisson(true_rate)[:, None].astype(float)
X = ((years - 1906.5) / 32.0)[:, None]                       # standardised time

kern = GPy.kern.Matern32(1)
kern.variance.set_prior(GPy.priors.LogGaussian(0.0, 1.0), warning=False)
kern.lengthscale.set_prior(GPy.priors.LogGaussian(0.0, 1.0), warning=False)
m = GPy.core.GP(X, Y, kernel=kern, likelihood=GPy.likelihoods.Poisson(),
                inference_method=GPy.inference.latent_function_inference.Laplace())

Xs = np.linspace(X.min(), X.max(), 150)[:, None]
years_s = Xs[:, 0] * 32.0 + 1906.5
true_s = 0.9 + 2.2 / (1 + np.exp((years_s - 1890) / 4.0))

plt.figure(figsize=(8, 3))
plt.bar(years, Y[:, 0], color="0.75", label="counts")
plt.plot(years_s, true_s, "k", label="true rate")
plt.legend(); plt.xlabel("year"); plt.show()

## INLA-style fit

With two hyperparameters `auto` chooses a grid design.

In [ ]:
t0 = time.time()
inla = INLAHyperparameters(m).fit()
t_inla = time.time() - t0
names = list(inla.hyperparameter_summary())
print(f"design={inla.design_used}, points={len(inla.weights)}, time={t_inla:.1f}s")
print("MAP:", dict(zip(m.parameter_names_flat(), np.round(m.unfixed_param_array, 3))))
mu_map, var_map = m.predict_noiseless(Xs)
mu_i, var_i = inla.predict_noiseless(Xs)

## HMC over the hyperparameters

The sampler starts at the mode with the Hessian as mass matrix. We discard the first 100 draws.

In [ ]:
np.random.seed(0)
u_star = inla.u_star.copy()
hmc = GPy.inference.mcmc.HMC(m, M=inla.hessian, stepsize=0.4)
t0 = time.time()
S = hmc.sample(num_samples=1500, hmc_iters=8)[100:]
t_hmc = time.time() - t0
accept = np.mean(np.any(np.diff(S, axis=0) != 0, axis=1))
print(f"HMC: {len(S)} draws kept, acceptance={accept:.2f}, time={t_hmc:.0f}s")
m.optimizer_array = u_star

## Hyperparameter marginals

We compare the HMC histogram with the weighted INLA design points, and report posterior means and standard deviations.

In [ ]:
vals = []
for u in inla.U:
    m.optimizer_array = u
    vals.append(np.array(m.unfixed_param_array, dtype=float))
m.optimizer_array = u_star
vals = np.array(vals)
w = inla.weights

fig, axes = plt.subplots(1, 3, figsize=(14, 3.2))
for j in range(2):
    ax = axes[j]
    bins = np.geomspace(min(S[:, j].min(), vals[:, j].min()), max(S[:, j].max(), vals[:, j].max()), 30)
    ax.hist(S[:, j], bins=bins, density=True, alpha=0.5, label="HMC")
    ax.hist(vals[:, j], bins=bins, weights=w, density=True, histtype="step", lw=2, label="INLA design")
    ax.axvline(m.unfixed_param_array[j], color="k", ls=":", label="MAP")
    ax.set_xscale("log"); ax.set_title(names[j])
axes[0].legend()
axes[2].plot(np.log(S[:, 0]), np.log(S[:, 1]), ".", alpha=0.2, label="HMC")
axes[2].scatter(np.log(vals[:, 0]), np.log(vals[:, 1]), s=300 * w / w.max(), c="C1", alpha=0.7, label="INLA design (size = weight)")
axes[2].set_xlabel("log variance"); axes[2].set_ylabel("log lengthscale"); axes[2].legend()
plt.tight_layout(); plt.show()

for j, n in enumerate(names):
    mi = np.sum(w * vals[:, j]); si = np.sqrt(np.sum(w * (vals[:, j] - mi) ** 2))
    print(f"{n:24s} INLA mean {mi:.3f} sd {si:.3f}   HMC mean {S[:, j].mean():.3f} sd {S[:, j].std():.3f}")

## Posterior for the log-rate

For HMC we average Laplace predictions over 250 thinned draws and combine variances with the law of total variance, as the INLA design does. The MAP plug-in ignores hyperparameter uncertainty.

In [ ]:
idx = np.linspace(0, len(S) - 1, 250).astype(int)
mus, vs = [], []
for k in idx:
    m.kern.variance = S[k, 0]; m.kern.lengthscale = S[k, 1]
    a, b = m.predict_noiseless(Xs)
    mus.append(a); vs.append(b)
m.optimizer_array = u_star
mus, vs = np.array(mus), np.array(vs)
mu_h = mus.mean(0)
var_h = (vs + mus**2).mean(0) - mu_h**2

fig, axes = plt.subplots(1, 2, figsize=(13, 3.6))
ax = axes[0]
for mu, v, lab, col in [(mu_map, var_map, "MAP", "k"), (mu_i, var_i, "INLA design", "C1"), (mu_h, var_h, "HMC", "C0")]:
    mu, v = mu.ravel(), v.ravel()
    ax.plot(years_s, np.exp(mu), color=col, label=lab, ls=":" if lab == "MAP" else "-")
    ax.fill_between(years_s, np.exp(mu - 2 * np.sqrt(v)), np.exp(mu + 2 * np.sqrt(v)), color=col, alpha=0.12)
ax.plot(years_s, true_s, "r--", label="truth"); ax.plot(years, Y, "k.", alpha=0.3)
ax.set_title("rate exp(f) with 2 sd bands on f"); ax.legend()
axes[1].plot(years_s, np.sqrt(var_map).ravel(), "k:", label="MAP"); axes[1].plot(years_s, np.sqrt(var_i).ravel(), "C1", label="INLA design"); axes[1].plot(years_s, np.sqrt(var_h).ravel(), "C0--", label="HMC")
axes[1].set_title("posterior sd of f"); axes[1].legend()
plt.tight_layout(); plt.show()

worst = lambda a, b: float(np.abs(a - b).max())
print(f"max |INLA - HMC|: mean of f {worst(mu_i, mu_h):.3f}, sd of f {worst(np.sqrt(var_i), np.sqrt(var_h)):.3f}")
print(f"max |MAP  - HMC|: mean of f {worst(mu_map, mu_h):.3f}, sd of f {worst(np.sqrt(var_map), np.sqrt(var_h)):.3f}")
print(f"cost: INLA {t_inla:.1f}s ({len(w)} Laplace fits plus Hessian), HMC {t_hmc:.0f}s")

## Reading the result

On this problem the INLA design and HMC agree on the latent posterior to about 0.003 in the mean and standard deviation of $f$, against about 0.025 for the MAP plug-in. So the design recovers most of what the plug-in misses, at under a second against roughly a minute for HMC. The difference from the plug-in is small in absolute terms because 112 counts pin the rate down well; hyperparameter uncertainty matters more with less data.

The hyperparameter marginals agree less closely than the predictions. The lengthscale matches well, but the variance standard deviation is about 0.81 under the design and 0.95 under HMC. The variance posterior has a heavy right tail, and the grid cutoff truncates it. The same effect is documented in CIP-0015, where a wider `grid_cutoff` or the importance design recovers the tail. The predictions are insensitive to it here because the tail has little posterior mass.

This is where INLA is strongest: two or three hyperparameters and a moderately sized latent field. See `cip/cip0015/check_high_dim.py` and CIP-0015 in the [GPy repository](https://github.com/SheffieldML/GPy) for higher dimensions, where the grid becomes expensive and the importance design takes over.

Limits of this comparison. Both methods share the Laplace approximation to $p(y\mid\theta)$, so neither tells us whether Laplace is accurate for these counts. Rue et al. address that with a simplified Laplace correction to the marginals of $f$, which `INLAHyperparameters` does not implement. A full test would compare against MCMC over $(f,\theta)$ jointly, for example elliptical slice sampling for $f$ within a hyperparameter sampler.